# Live demo: Hybrid Physics–ML PV model under partial shading

**What this shows.** A 12-panel array has 3 strings of 4 panels. You choose the sunlight, the temperature and how much each panel is shaded. Four estimates of the array's power are then computed **live**:

| Model | What it is |
|---|---|
| **PVMismatch ("true")** | Cell-level simulation with bypass diodes. This is the reference. |
| **Physics (pvlib)** | Single-diode model that only sees the *average* sunlight, so it cannot see *which* panels are shaded. |
| **Pure ML** | XGBoost trained to predict power directly. |
| **Hybrid** | Physics plus an XGBoost correction of the physics error, using per-string inputs. This is our model. |

The two ML models were trained only on in-distribution rows (G 200–1000 W/m², T 10–45 °C) of the 5,000-row dataset.

**How to run:** use *Run → Run All Cells*, then use the controls at the bottom.


In [ ]:
import json, warnings
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display, clear_output
from xgboost import XGBRegressor
from pv_sim import pvmismatch_power, pvlib_power, features
warnings.filterwarnings("ignore")

pure_ml = XGBRegressor(); pure_ml.load_model("models/pure_ml.json")
hybrid  = XGBRegressor(); hybrid.load_model("models/hybrid_residual.json")

def predict_all(G, T, mask):
    X = features(G, T, mask)
    p_true = pvmismatch_power(G, T, mask)
    p_phys = pvlib_power(G, T, mask)
    return {"PVMismatch (true)": p_true,
            "Physics (pvlib)":   p_phys,
            "Pure ML":           float(pure_ml.predict(X)[0]),
            "Hybrid (ours)":     p_phys + float(hybrid.predict(X)[0])}

print("Models loaded. Quick check, no shading at 1000 W/m², 25 °C:")
for k, v in predict_all(1000, 25, np.ones(12)).items():
    print(f"  {k:<18} {v:7.0f} W")

## Key comparison: *where* the shade falls matters

Both cases below shade **four panels by 60%**, so the average sunlight is identical and pvlib gives the **same** answer for both. The real array loses about 550 W more when the shade is spread across strings, because every string then has a weak panel and the bypass diodes and string currents respond differently. The hybrid follows the true value in both cases (within about 30–60 W). The last two lines show an **OOD** case, at 1150 W/m², which is above the training range.

In [ ]:
def mask_from(panels, sev):
    m = np.ones(12)
    for p in panels: m[p - 1] = 1 - sev
    return m

cases = [("4 panels, ONE string (P1-P4)",       800, 25, mask_from([1, 2, 3, 4], 0.6)),
         ("4 panels, SPREAD (P1,P2,P5,P9)",     800, 25, mask_from([1, 2, 5, 9], 0.6)),
         ("OOD G=1150, no shade",              1150, 25, np.ones(12)),
         ("OOD G=1150, 2 panels (P1,P2) 70%",  1150, 25, mask_from([1, 2], 0.7))]
print(f"{'case':<36}{'G':>6}{'true':>8}{'physics':>9}{'pure ML':>9}{'hybrid':>8}   (W)")
for name, g, t, m in cases:
    r = predict_all(g, t, m)
    print(f"{name:<36}{g:>6}" + "".join(f"{v:>9.0f}" for v in r.values()))

## Interactive array

The 12 sliders follow the physical layout: **each row is one string**. A slider value is the fraction of sunlight the panel receives, so 1.0 means unshaded and 0.3 means 70% shaded. Use the presets for a quick start.

Values outside the training range (G above 1000 or below 200, T above 45 or below 10) are flagged **OOD**. This is where pure ML tends to fail and the hybrid keeps working.

In [ ]:
COLORS = {"PVMismatch (true)": "#0b0b0b", "Physics (pvlib)": "#7a7974",
          "Pure ML": "#eb6834", "Hybrid (ours)": "#2a78d6"}
PRESETS = {
    "No shading":                          ([], 0.0),
    "1 panel 70% (P6)":                    ([6], 0.7),
    "2 panels same string 70% (P1,P2)":    ([1, 2], 0.7),
    "2 panels different strings 70% (P1,P5)": ([1, 5], 0.7),
    "4 panels in one string 60% (P1-P4)":  ([1, 2, 3, 4], 0.6),
    "4 panels spread 60% (P1,P2,P5,P9)":   ([1, 2, 5, 9], 0.6),
    "Uniform 50% (all)":                   (list(range(1, 13)), 0.5),
}
style = {"description_width": "initial"}
G = W.FloatSlider(800, min=100, max=1200, step=25, description="Irradiance G (W/m²)", style=style,
                  continuous_update=False, layout=W.Layout(width="420px"))
T = W.FloatSlider(25, min=0, max=60, step=1, description="Ambient temp T (°C)", style=style,
                  continuous_update=False, layout=W.Layout(width="420px"))
S = [W.FloatSlider(1.0, min=0.1, max=1.0, step=0.05, description=f"P{i+1}",
                   continuous_update=False, readout_format=".2f", layout=W.Layout(width="210px"))
     for i in range(12)]
preset = W.Dropdown(options=list(PRESETS), description="Preset", layout=W.Layout(width="420px"))
out = W.Output()

def apply_preset(change):
    panels, sev = PRESETS[change["new"]]
    for i, s in enumerate(S):
        s.value = 1 - sev if (i + 1) in panels else 1.0
preset.observe(apply_preset, names="value")

def ood_text(g, t):
    flags = [n for n, bad in [("G", g < 200 or g > 1000), ("T", t < 10 or t > 45)] if bad]
    return "OOD: " + "+".join(flags) if flags else "in-distribution"

def redraw(*_):
    mask = np.array([s.value for s in S])
    r = predict_all(G.value, T.value, mask)
    true = r["PVMismatch (true)"]
    with out:
        clear_output(wait=True)
        fig, (a0, a1) = plt.subplots(1, 2, figsize=(10, 3.4), gridspec_kw={"width_ratios": [1, 2.2]})
        im = a0.imshow(mask.reshape(3, 4), cmap="Blues", vmin=0, vmax=1)
        for i, v in enumerate(mask):
            a0.text(i % 4, i // 4, f"P{i+1}\n{v:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if v > 0.6 else "black")
        a0.set_xticks([]); a0.set_yticks(range(3), ["String 1", "String 2", "String 3"])
        a0.set_title("Sunlight fraction per panel", fontsize=9)
        names = list(r)
        bars = a1.barh(names[::-1], [r[n] for n in names[::-1]], color=[COLORS[n] for n in names[::-1]])
        for b, n in zip(bars, names[::-1]):
            err = "" if n.startswith("PVM") else f"  ({r[n]-true:+.0f} W, {100*(r[n]-true)/true:+.1f}%)"
            a1.text(b.get_width(), b.get_y() + b.get_height()/2, f" {r[n]:.0f} W{err}", va="center", fontsize=8)
        a1.set_xlim(0, max(r.values()) * 1.45); a1.set_xlabel("Array power (W)")
        a1.set_title(f"G={G.value:.0f} W/m², T={T.value:.0f} °C, {ood_text(G.value, T.value)}", fontsize=9)
        for s in ["top", "right"]: a1.spines[s].set_visible(False)
        plt.tight_layout(); plt.show()

for w in [G, T, *S]: w.observe(redraw, names="value")
grid = W.VBox([W.HBox(S[k*4:(k+1)*4]) for k in range(3)])
display(W.VBox([preset, G, T, W.HTML("<b>Panel sunlight fraction</b> (rows = strings)"), grid, out]))
redraw()

## Suggested 5-minute demo script

1. **No shading, 800 W/m², 25 °C.** All four models agree, so the physics model is calibrated.
2. **"1 panel 70% (P6)".** A single shaded panel already makes pvlib overestimate by about 375 W, because the bypass-diode loss is invisible to an average-sunlight model. The hybrid is within about 80 W.
3. **"4 panels in one string" vs "4 panels spread".** pvlib gives the *same* number (about 2260 W) for both, but the true power differs by about 550 W. The hybrid tracks both cases to within about 30–60 W. *This is the point of the project.*
4. **Push G to 1150 W/m² (OOD).** Pure ML stops increasing because tree models cannot extrapolate (error of about −400 to −550 W). The hybrid keeps following the physics trend (about 60–75 W off).
5. **Push T to 55 °C (OOD).** The hybrid still tracks the true value.
6. **Uniform 50% shade.** Physics alone is already good here, because every panel gets the same sunlight. Thanks to the per-string inputs, the hybrid stays just as good (within a few W). Without them it used to over-correct.
